In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("../../Datasets/evaluacion1.csv")

In [9]:
ev1 = pd.read_csv('../../Datasets/amazon_final.csv')
ev1.head(1)

,product_title,product_rating,total_reviews,purchased_last_month,original_price,is_sponsored,coupon,buy_box_availability,sustainability_tags,product_image_url,...,badge_amazons,badge_best_seller,badge_ends_in,badge_limited_time_deal,badge_no_badge,badge_save_xpct,product_category,log_purchased_last_month,log_original_price,log_total_reviews
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,358.0,0,239.99,1,No Coupon,1,0,https://m.media-amazon.com/images/I/51vvx1JymA...,...,0,0,0,0,1,0,PC Components,0.0,5.484755,5.883322


In [10]:
ev1.drop(columns=['product_title', 'total_reviews', 'purchased_last_month', 'coupon', 'product_image_url'], inplace=True)

In [13]:
# Aplicar One-Hot Encoding a las categorías
ev1 = pd.get_dummies(ev1, columns=['product_category'], prefix='cat')

In [14]:
ev1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 35 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   product_rating                                     7717 non-null   float64
 1   original_price                                     7717 non-null   float64
 2   is_sponsored                                       7717 non-null   int64  
 3   buy_box_availability                               7717 non-null   int64  
 4   sustainability_tags                                7717 non-null   int64  
 5   has_coupon                                         7717 non-null   int64  
 6   discount_percentage                                7717 non-null   float64
 7   badge_amazons                                      7717 non-null   int64  
 8   badge_best_seller                                  7717 non-null   int64  
 9   badge_en

In [22]:
# ============================================================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================================================
print("="*60)
print("PREPARACIÓN DEL DATASET PARA LASSO REGRESSION")
print("="*60)

# Cargar datos
df = ev1

# Variable objetivo: log_original_price (transformación logarítmica)
y = df['log_original_price']

# Features: Todas excepto las relacionadas con precio
X = df.drop(['original_price', 'log_original_price'], axis=1)

print(f"Dataset shape: {df.shape}")
print(f"Features: {X.shape[1]}")
print(f"Target: log_original_price (transformado)")
print(f"Original price range: €{df['original_price'].min():.2f} - €{df['original_price'].max():.2f}")

# ============================================================================
# 2. PREPROCESAMIENTO
# ============================================================================
# Separar train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"\nTrain size: {X_train.shape[0]} samples")
print(f"Test size: {X_test.shape[0]} samples")

# Identificar columnas numéricas continuas
numeric_cols = ['product_rating', 'discount_percentage', 
                'log_purchased_last_month', 'log_total_reviews']

# Verificar que existen
numeric_cols = [col for col in numeric_cols if col in X_train.columns]
print(f"\nNumeric columns to scale: {numeric_cols}")

# Escalar features numéricas
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# ============================================================================
# 3. ENTRENAMIENTO CON LASSO - BÚSQUEDA DE ALPHA ÓPTIMO
# ============================================================================
print("\n" + "="*60)
print("BUSCANDO ALPHA ÓPTIMO PARA LASSO")
print("="*60)

# Definir rango de alphas a probar
alphas = np.logspace(-5, 2, 50)  # 0.00001 a 100

# Usar GridSearchCV para encontrar el mejor alpha
lasso = Lasso(max_iter=10000, random_state=42)
param_grid = {'alpha': alphas}

grid_search = GridSearchCV(
    lasso, 
    param_grid, 
    cv=5, 
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_scaled, y_train)

print(f"Mejor alpha encontrado: {grid_search.best_params_['alpha']:.6f}")
print(f"Mejor score MAE (negativo): {grid_search.best_score_:.4f}")

# Entrenar modelo final con el mejor alpha
best_lasso = Lasso(
    alpha=grid_search.best_params_['alpha'],
    max_iter=10000,
    random_state=42
)

best_lasso.fit(X_train_scaled, y_train)

# ============================================================================
# 4. PREDICCIÓN
# ============================================================================
print("\n" + "="*60)
print("REALIZANDO PREDICCIONES")
print("="*60)

# --- 6. EVALUACIÓN --- (EXACTAMENTE COMO PIDES)
y_pred_log = best_lasso.predict(X_test_scaled)
y_test_euro = np.expm1(y_test)
y_pred_euro = np.expm1(y_pred_log)

print(f"Predicciones realizadas:")
print(f"- y_pred_log shape: {y_pred_log.shape}")
print(f"- y_test_euro shape: {y_test_euro.shape}")
print(f"- y_pred_euro shape: {y_pred_euro.shape}")

# ============================================================================
# 5. EVALUACIÓN COMPLETA
# ============================================================================
print("\n" + "="*60)
print("EVALUACIÓN DEL MODELO LASSO")
print("="*60)

# 5.1 Métricas en escala logarítmica
print("\n1. MÉTRICAS EN ESCALA LOGARÍTMICA:")
print("-" * 40)

mae_log = mean_absolute_error(y_test, y_pred_log)
mse_log = mean_squared_error(y_test, y_pred_log)
rmse_log = np.sqrt(mse_log)
r2_log = r2_score(y_test, y_pred_log)

print(f"MAE (log):  {mae_log:.4f}")
print(f"MSE (log):  {mse_log:.4f}")
print(f"RMSE (log): {rmse_log:.4f}")
print(f"R² (log):   {r2_log:.4f}")

# 5.2 Métricas en Euros (escala original)
print("\n2. MÉTRICAS EN EUROS (ESCALA ORIGINAL):")
print("-" * 40)

mae_euro = mean_absolute_error(y_test_euro, y_pred_euro)
mse_euro = mean_squared_error(y_test_euro, y_pred_euro)
rmse_euro = np.sqrt(mse_euro)
r2_euro = r2_score(y_test_euro, y_pred_euro)

# Error porcentual
errors_euro = y_pred_euro - y_test_euro
mape = np.mean(np.abs(errors_euro / y_test_euro)) * 100

print(f"MAE (€):    €{mae_euro:.2f}")
print(f"RMSE (€):   €{rmse_euro:.2f}")
print(f"R² (€):     {r2_euro:.4f}")
print(f"MAPE:       {mape:.2f}%")

# 5.3 Estadísticas del error
print("\n3. ESTADÍSTICAS DEL ERROR (EUROS):")
print("-" * 40)

error_stats = {
    'Error medio': errors_euro.mean(),
    'Desviación estándar': errors_euro.std(),
    'Error mínimo': errors_euro.min(),
    'Error máximo': errors_euro.max(),
    'Error absoluto medio': np.abs(errors_euro).mean(),
    'Error relativo medio (%)': (errors_euro / y_test_euro).mean() * 100
}

for stat, value in error_stats.items():
    if '€' not in stat and '%' not in stat:
        print(f"{stat:25}: €{value:.2f}")
    else:
        print(f"{stat:25}: {value:.2f}")

# 5.4 Distribución de errores porcentuales
print("\n4. DISTRIBUCIÓN DE ERRORES PORCENTUALES:")
print("-" * 40)

error_percentages = (errors_euro / y_test_euro) * 100
abs_error_percentages = np.abs(error_percentages)

bins = [0, 5, 10, 20, 50, 100, float('inf')]
labels = ['<5%', '5-10%', '10-20%', '20-50%', '50-100%', '>100%']

dist = pd.cut(abs_error_percentages, bins=bins, labels=labels, right=False)
dist_counts = dist.value_counts().sort_index()
dist_percent = (dist_counts / len(dist_counts) * 100).round(2)

for bin_label, count, percent in zip(dist_counts.index, dist_counts.values, dist_percent.values):
    print(f"{bin_label:10}: {count:4d} muestras ({percent:5.1f}%)")

# ============================================================================
# 6. ANÁLISIS DE COEFICIENTES LASSO
# ============================================================================
print("\n" + "="*60)
print("ANÁLISIS DE COEFICIENTES LASSO")
print("="*60)

# Crear DataFrame de coeficientes
coef_df = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': best_lasso.coef_,
    'Abs_Coeficiente': np.abs(best_lasso.coef_)
})

# Ordenar por importancia absoluta
coef_df = coef_df.sort_values('Abs_Coeficiente', ascending=False)

# Calcular impacto aproximado en precio
# Para interpretación: cambio de 1 std en feature → cambio % en precio
coef_df['Impacto_Aprox_%'] = (np.exp(coef_df['Coeficiente']) - 1) * 100

# Identificar variables eliminadas (coeficiente = 0)
zero_mask = np.abs(coef_df['Coeficiente']) < 1e-10
zero_coef_vars = coef_df[zero_mask]
non_zero_coef_vars = coef_df[~zero_mask]

print(f"\nResumen de coeficientes:")
print(f"• Total variables: {len(coef_df)}")
print(f"• Variables con coeficiente ≠ 0: {len(non_zero_coef_vars)}")
print(f"• Variables eliminadas (coef ≈ 0): {len(zero_coef_vars)}")
print(f"• Porcentaje eliminado: {len(zero_coef_vars)/len(coef_df)*100:.1f}%")

# 6.1 Variables más importantes
print("\nTOP 15 VARIABLES MÁS IMPORTANTES:")
print("-" * 70)

top_n = 15
for idx, row in non_zero_coef_vars.head(top_n).iterrows():
    impacto = f"+{row['Impacto_Aprox_%']:.1f}%" if row['Impacto_Aprox_%'] > 0 else f"{row['Impacto_Aprox_%']:.1f}%"
    print(f"{row['Variable']:55} : {row['Coeficiente']:+.6f} ({impacto})")

# 6.2 Variables eliminadas
if len(zero_coef_vars) > 0:
    print(f"\nPRIMERAS 10 VARIABLES ELIMINADAS (coef = 0):")
    print("-" * 70)
    
    for idx, row in zero_coef_vars.head(10).iterrows():
        print(f"{row['Variable']:55} : {row['Coeficiente']:.6f}")

# 6.3 Análisis por tipo de variable
print("\nANÁLISIS POR TIPO DE VARIABLE:")
print("-" * 70)

coef_df['Tipo'] = 'Otro'
coef_df.loc[coef_df['Variable'].str.startswith('cat_'), 'Tipo'] = 'Categoría'
coef_df.loc[coef_df['Variable'].str.startswith('badge_'), 'Tipo'] = 'Badge'
coef_df.loc[coef_df['Variable'].str.contains('log_'), 'Tipo'] = 'Log Feature'
coef_df.loc[coef_df['Variable'].str.contains('rating|discount'), 'Tipo'] = 'Métrica'

type_summary = coef_df.groupby('Tipo').agg({
    'Variable': 'count',
    'Coeficiente': lambda x: np.sum(x != 0),
    'Abs_Coeficiente': 'mean'
}).round(4)

type_summary = type_summary.rename(columns={
    'Variable': 'Total',
    'Coeficiente': 'No_Cero',
    'Abs_Coeficiente': 'Coef_Promedio'
})

type_summary['%_Retenidas'] = (type_summary['No_Cero'] / type_summary['Total'] * 100).round(1)
print(type_summary)

# ============================================================================
# 7. EJEMPLOS DE PREDICCIÓN
# ============================================================================
print("\n" + "="*60)
print("EJEMPLOS CONCRETOS DE PREDICCIÓN")
print("="*60)

# Seleccionar ejemplos representativos
np.random.seed(42)
n_examples = 10
sample_indices = np.random.choice(len(y_test), n_examples, replace=False)

print(f"\n{'Índex':>6} {'Real (€)':>12} {'Predicho (€)':>14} {'Error (€)':>12} {'Error %':>10} {'Calidad':>10}")
print("-" * 70)

for i, idx in enumerate(sample_indices, 1):
    real = y_test_euro.iloc[idx] if hasattr(y_test_euro, 'iloc') else y_test_euro[idx]
    pred = y_pred_euro[idx]
    error = pred - real
    error_pct = (error / real) * 100
    
    # Clasificar calidad de predicción
    if abs(error_pct) <= 10:
        calidad = "Excelente"
    elif abs(error_pct) <= 20:
        calidad = "Buena"
    elif abs(error_pct) <= 50:
        calidad = "Aceptable"
    else:
        calidad = "Mala"
    
    print(f"{idx:6d} {real:12.2f} {pred:14.2f} {error:12.2f} {error_pct:10.1f}% {calidad:>10}")

# ============================================================================
# 8. GUARDAR RESULTADOS
# ============================================================================
print("\n" + "="*60)
print("GUARDANDO RESULTADOS")
print("="*60)

# 8.1 Guardar coeficientes
coef_df.to_csv('lasso_coeficientes_detallados.csv', index=False)
print(f"✓ Coeficientes guardados en: lasso_coeficientes_detallados.csv")

# 8.2 Guardar variables eliminadas
if len(zero_coef_vars) > 0:
    zero_coef_vars[['Variable', 'Coeficiente']].to_csv('lasso_variables_eliminadas.csv', index=False)
    print(f"✓ Variables eliminadas guardadas en: lasso_variables_eliminadas.csv")

# 8.3 Guardar predicciones
predictions_df = pd.DataFrame({
    'index_original': X_test.index,
    'y_test_log': y_test.values,
    'y_pred_log': y_pred_log,
    'y_test_euro': y_test_euro.values,
    'y_pred_euro': y_pred_euro,
    'error_euro': errors_euro,
    'error_percent': error_percentages
})

predictions_df.to_csv('lasso_predicciones_completas.csv', index=False)
print(f"✓ Predicciones guardadas en: lasso_predicciones_completas.csv")

# 8.4 Guardar métricas
metrics_dict = {
    'alpha_optimo': grid_search.best_params_['alpha'],
    'mae_log': mae_log,
    'rmse_log': rmse_log,
    'r2_log': r2_log,
    'mae_euro': mae_euro,
    'rmse_euro': rmse_euro,
    'r2_euro': r2_euro,
    'mape': mape,
    'n_variables_totales': X.shape[1],
    'n_variables_seleccionadas': len(non_zero_coef_vars),
    'porcentaje_eliminado': len(zero_coef_vars)/len(coef_df)*100,
    'train_samples': X_train.shape[0],
    'test_samples': X_test.shape[0]
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv('lasso_metricas_evaluacion.csv', index=False)
print(f"✓ Métricas guardadas en: lasso_metricas_evaluacion.csv")

# ============================================================================
# 9. RESUMEN PARA COMPARTIR
# ============================================================================
print("\n" + "="*60)
print("RESUMEN EJECUTIVO - LASSO BASELINE")
print("="*60)

print(f"\n📊 PERFORMANCE DEL MODELO:")
print(f"   • MAE en Euros: €{mae_euro:.2f}")
print(f"   • RMSE en Euros: €{rmse_euro:.2f}")
print(f"   • R² en Euros: {r2_euro:.4f}")
print(f"   • Error porcentual medio (MAPE): {mape:.1f}%")

print(f"\n🔍 SELECCIÓN DE FEATURES (LASSO):")
print(f"   • Alpha óptimo: {grid_search.best_params_['alpha']:.6f}")
print(f"   • Variables totales: {X.shape[1]}")
print(f"   • Variables seleccionadas: {len(non_zero_coef_vars)}")
print(f"   • Variables eliminadas: {len(zero_coef_vars)}")
print(f"   • Tasa de eliminación: {len(zero_coef_vars)/len(coef_df)*100:.1f}%")

print(f"\n🏆 TOP 5 VARIABLES MÁS IMPORTANTES:")
top5 = non_zero_coef_vars.head(5)
for i, (_, row) in enumerate(top5.iterrows(), 1):
    impacto = f"+{row['Impacto_Aprox_%']:.1f}%" if row['Impacto_Aprox_%'] > 0 else f"{row['Impacto_Aprox_%']:.1f}%"
    print(f"   {i}. {row['Variable']}: {impacto}")

print(f"\n📁 ARCHIVOS GENERADOS:")
print(f"   1. lasso_coeficientes_detallados.csv - Todos los coeficientes")
print(f"   2. lasso_variables_eliminadas.csv - Variables para simplificar modelos")
print(f"   3. lasso_predicciones_completas.csv - Predicciones detalladas")
print(f"   4. lasso_metricas_evaluacion.csv - Métricas del baseline")

print(f"\n💡 RECOMENDACIONES PARA BEJA Y DIEGO:")
print(f"   1. Baseline establecido: MAE = €{mae_euro:.2f}")
print(f"   2. Pueden eliminar {len(zero_coef_vars)} variables de sus modelos")
print(f"   3. Variables clave identificadas: {len(non_zero_coef_vars)} features importantes")


PREPARACIÓN DEL DATASET PARA LASSO REGRESSION
Dataset shape: (7717, 35)
Features: 33
Target: log_original_price (transformado)
Original price range: €2.16 - €5449.00

Train size: 6173 samples
Test size: 1544 samples

Numeric columns to scale: ['product_rating', 'discount_percentage', 'log_purchased_last_month', 'log_total_reviews']

BUSCANDO ALPHA ÓPTIMO PARA LASSO
Mejor alpha encontrado: 0.000010
Mejor score MAE (negativo): -0.7852

REALIZANDO PREDICCIONES
Predicciones realizadas:
- y_pred_log shape: (1544,)
- y_test_euro shape: (1544,)
- y_pred_euro shape: (1544,)

EVALUACIÓN DEL MODELO LASSO

1. MÉTRICAS EN ESCALA LOGARÍTMICA:
----------------------------------------
MAE (log):  0.7564
MSE (log):  0.9077
RMSE (log): 0.9527
R² (log):   0.4671

2. MÉTRICAS EN EUROS (ESCALA ORIGINAL):
----------------------------------------
MAE (€):    €150.13
RMSE (€):   €355.79
R² (€):     0.3194
MAPE:       111.40%

3. ESTADÍSTICAS DEL ERROR (EUROS):
----------------------------------------
Error m

In [3]:
df_lasso_coeficientes = pd.read_csv("lasso_coeficientes_detallados.csv")
df_lasso_coeficientes.head(10)


,Variable,Coeficiente,Abs_Coeficiente,Impacto_Aprox_%,Tipo
0,cat_Power & Batteries,-1.030179,1.030179,-64.305676,Categoría
1,"cat_Chargers, Adapters & Cables",-0.905472,0.905472,-59.564897,Categoría
2,badge_save_xpct,0.825802,0.825802,128.371102,Badge
3,cat_Small Gadget Accessories (Cases & Protectors),-0.697781,0.697781,-50.231156,Categoría
4,cat_Laptops,0.682571,0.682571,97.895899,Categoría
5,cat_Printers & Scanners (Hardware),0.679448,0.679448,97.278788,Categoría
6,cat_Cameras & Photography,0.595981,0.595981,81.480964,Categoría
7,is_sponsored,-0.477953,0.477953,-37.994864,Otro
8,log_purchased_last_month,-0.367792,0.367792,-30.773918,Log Feature
9,cat_TV & Video Displays,0.354768,0.354768,42.584968,Categoría


- 1.Coeficiente(Peso en el modelo Lasso) = Positivo → Aumenta el precio predicho / Negativo → Disminuye el precio predicho / Valor absoluto mayor → Mayor importancia
- 2.Impacto_Aprox_% = Impacto porcentual aproximado en el precio

In [5]:
df_lasso_metricas = pd.read_csv("lasso_metricas_evaluacion.csv")
df_lasso_metricas.head()


,alpha_optimo,mae_log,rmse_log,r2_log,mae_euro,rmse_euro,r2_euro,mape,n_variables_totales,n_variables_seleccionadas,porcentaje_eliminado,train_samples,test_samples
0,0.00001,0.756371,0.95273,0.467145,150.131022,355.785694,0.319425,111.396197,33,30,9.090909,6173,1544


- 1.alpha_optimo = Indica que los datos necesitan poca regularización
- 2.mae_log(MÉTRICAS EN ESCALA LOGARÍTMICA (predicción del log(precio))) =  tus predicciones del log(precio) se equivocan por ±0.756 unidades log. Ejemplo: Si el precio real = log(100) = 4.605, predicción estaría entre 3.849 y 5.361
- 3.rmse_log(Raíz del Error Cuadrático Medio en escala log) = Penaliza más los errores grandes / Más alto que MAE → hay outliers/errores grandes
- 4.r2_log(Coeficiente de determinación en escala log) = El modelo explica el 46.7% de la varianza en log(precios)
- 5.mae_euro(Error Absoluto Medio en Euros)  = En promedio, se equivocas por ±€150 al predecir el precio.Con precios de €2-€5449: es un error alto
- 6.rmse_euro(Raíz del Error Cuadrático Medio en Euros)  = Casi €356 de error promedio (penaliza errores grandes).Significativamente mayor que MAE → hay predicciones muy malas
- 7.r2_euro = explica el 31.9% de la varianza en precios originales. / BAJO: El modelo no captura bien las variaciones de precio / Posible causa: outliers extremos
- 8.mape(Error Porcentual Absoluto Medio) = CRÍTICO: ¡Error promedio del 111%!
- 9.n_variables_totales = Total de features/predictores disponibles
- 10.n_variables_seleccionadas  = 30 de 33 variables aportan algo al modelo
- 11.porcentaje_eliminado = Solo el 9.1% de variables eliminadas / Lasso encontró que casi todo es relevante

In [6]:
df_lasso_prediccion = pd.read_csv("lasso_predicciones_completas.csv")
df_lasso_prediccion.head(10)

,index_original,y_test_log,y_pred_log,y_test_euro,y_pred_euro,error_euro,error_percent
0,2168,3.610648,6.254974,35.99,519.595777,483.605777,1343.722636
1,472,4.905201,3.672871,133.99,38.364770,-95.625230,-71.367438
2,4159,4.453882,4.838322,84.96,125.257262,40.297262,47.430864
3,676,3.483392,3.867315,31.57,46.813843,15.243843,48.285851
4,6659,7.261927,6.444618,1424.00,628.306153,-795.693847,-55.877377
5,2685,5.991465,4.677128,399.00,106.460951,-292.539049,-73.318057
6,3185,4.642562,5.647799,102.81,282.666529,179.856529,174.940695
7,5830,6.552494,5.573476,699.99,262.347858,-437.642142,-62.521199
8,2892,3.713328,4.303659,39.99,72.969954,32.979954,82.470502
9,3298,5.935476,5.260822,377.22,191.639856,-185.580144,-49.196793


In [8]:
df_lasso_variables = pd.read_csv("lasso_variables_eliminadas.csv")
df_lasso_variables.head(10)

,Variable,Coeficiente
0,badge_limited_time_deal,-0.0
1,cat_Smart Home & Security,-0.0
2,cat_Video Game Consoles & Virtual Reality,-0.0
